# Gemma 4: A 16384 Output Cap Refuses Long Prompts

The host's sample `configs/sampling.yaml` asks for `max_output_tokens: 16384`, and the official starter writes the same
value when it rewrites that file. The scorer serves Gemma with vLLM at `max_model_len = 32768`, and vLLM refuses any
call whose prompt plus requested output is more than 32,768 tokens. So with the sample config, **every call whose
prompt reaches 16,385 tokens is refused**; in the official starter's evaluator the task then ends with no patch.

I ran the sample agent on 7 public development tasks on 4 x L4, one task at a time, and kept every trace. The prompts
and `sampling.yaml` were the host's (checked by SHA-256 in section 1); the budgets were raised to 120 s per command, 30
tool calls, 4 minutes and 60 turns, and the 2 placeholder LoRA adapters were removed. **1 of the 7 traced tasks was
refused this way**, at its 12th model call, 143.4 s in, before it had edited anything. Another peaked at 15,813
tokens, 571 below the line.

The rule itself is public already: [Oleksii Zhukov's walkthrough](https://www.kaggle.com/code/zhukovoleksiy/gemma-4-walkthrough-first-submission)
states it, and the top public bundle runs at 8,192 ([Roman Rozen](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1)).
What these traces add is the measurement, and why the harness's own safety net does not catch it:

* **Compaction leaves at most 2,048 tokens of room.** The harness compacts the history once a prompt reaches 14,336
  tokens, 2,048 below the 16,384 ceiling. It acts on the prompt just sent, not on prompt + output, so it is one call
  late. All 4 prompts at or above 14,336 that had a next call were followed by a compacted one (the fifth was
  httpx_3672's last call), and none of the 97 below it was. 3 of 97 steps added more than 2,048 tokens, but they
  started far below the trigger. The fatal one started at 12,597 with 3,787 tokens left and added at least 3,788 (one
  `read_file` reply of 10,848 characters) before compaction could react.
* **The knob: set `max_output_tokens: 8192`.** The ceiling moves to 24,576, 10,240 above the trigger, 1.8x the
  densest-rate estimate of the fatal step. Replayed over every traced call it refuses nothing and cuts no output (the
  largest output was 3,088 tokens). The README lists the trigger as a fixed 14,336 (not tested here at another cap),
  so a cap below 8,192 buys no extra working history: only room for a bigger single step, at the cost of output room.
  No cap covers an uncapped `search_similar_code` result (section 3), and prvsiyan reports a refusal even at a 2,048
  cap.
* **Time is spent generating.** Model calls took 98.7% to 100.0% of each task's agent time (34.0 output tokens/s,
  prefill included, one task at a time), 90% of the generated text was thinking that the next call never sees, and
  each compaction cost 24 to 33 s. The 12-hour limit allows a `max_time_minutes` of at most 4.94 (120 tasks, 60 s over
  budget per task) to 5.98 (116 tasks, 10 s over), one task at a time; section 5 has the caveats.

Sections: 1 the trap in the host's files, 2 the 7 tasks, 3 why compaction misses it, 4 the fix with a calculator and a
replay, 5 the time budget, 6 the cost table, 7 limits and credits.

> **Update, Oct 1:** I submitted exactly this change (8,192, with the 4-minute budgets used here). The scorer returned
> an error with no score, and it does not say why. The same agent at 16,384 had scored 0.06 (section 2). My best guess
> is the 12-hour limit: tasks that used to be refused early now run their full time. Check section 5's time arithmetic,
> or lower `max_time_minutes`, before you make this change. I'll update this when a test settles it.

## 0. Setup and the knobs

Nothing here runs a model. The measurements are tables stored in the next cell (per-call token counts from my own
traces, per-task outcomes, GPU samples). Quoted lines come in two kinds, and each is labelled where it is printed: the
host's lines (`HARNESS_README.md`, the sample's `sampling.yaml` and prompts) are CHECKED against the mounted competition
data when it is attached; my run-log lines, the official starter's lines, the Evaluation and Data page sentences, my
public score and figures from other notebooks are COPIED. The knobs sit at the top of the next cell; change the values
to your own config:

* `MAX_OUTPUT_TOKENS`, `THINKING_BUDGET`: your `configs/sampling.yaml`. Section 4 says whether that cap is safe.
* `LARGEST_STEP_TOKENS`: the most one tool round trip adds to your agent's prompt, if you have measured it. `None` uses
  an estimate from these traces.
* `MAX_TURNS`, `MAX_TIME_MINUTES`, `N_HIDDEN_TASKS`: your `eval_config.yaml` budgets and the hidden task count, for the
  turn-depth and time arithmetic.

What ran: the host's `sample_submission` with its `eval_config.yaml` budgets raised and its 2 placeholder LoRA adapters
removed, evaluated by the official starter's own evaluator. The starter's unmodified 2-task run, at the sample's
1-minute budget, is stored as well.

In [1]:
import base64, gzip, hashlib, io, json, math, os, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.stats import beta

INPUT = Path(os.environ.get("SMOKE_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("SMOKE_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
T0 = time.time()

# ---- knobs: change these to your own configuration -------------------------------------------
MAX_OUTPUT_TOKENS = 8_192      # configs/sampling.yaml max_output_tokens (the host sample ships 16,384)
THINKING_BUDGET = 4_096        # thinking_config.thinking_budget; 0 if thinking is off
LARGEST_STEP_TOKENS = None     # the most one tool round trip adds to the prompt; None = estimated from these traces
MAX_TURNS = 60                 # eval_config.yaml max_turns (model calls per task)
MAX_TIME_MINUTES = 4           # eval_config.yaml max_time_minutes
N_HIDDEN_TASKS = 116           # 58 public + 58 private (section 5)

# ---- set by the scorer, not by your submission ------------------------------------------------
CONTEXT = 32_768               # vLLM max_model_len
COMPACTION_TRIGGER = 14_336    # EventsCompactionConfig token_threshold

HL = {}                        # headline numbers, as printed


def rec(key, value):
    HL[key] = value
    return value


def n(x):
    """Integer with a thousands separator, the way the text quotes it."""
    return f"{int(round(float(x))):,}"


# ---- the measurements, stored in this notebook ------------------------------------------------
# One row per answered model call, from my own traces of the host's sample agent (run "dev": 7 public development
# tasks; run "base": the official starter's own 2-task run). Counts only: no task text, no patches, no repository
# content. t_s = seconds since the task's first trace event; tool_done_s = when that call's tool returned;
# obs_chars = length of the tool's JSON reply as recorded; read_content_chars = length of the file text inside a
# read_file reply.
CALLS_CSV = """run,task,call,t_s,tool_done_s,prompt_tokens,completion_tokens,tool,args_chars,obs_chars,thinking_chars,read_content_chars,read_truncated
dev,fastapi_15588,1,4.82,5.95,5442,95,search_similar_code,31,74,301,0,
dev,fastapi_15588,2,7.41,7.44,5504,51,search_similar_code,16,1266,112,0,
dev,fastapi_15588,3,8.59,8.78,6008,26,run_command,40,5891,0,0,
dev,fastapi_15588,4,14.67,14.68,8378,149,read_file,30,5079,495,4684,True
dev,fastapi_15588,5,58.28,58.28,9944,1481,read_file,33,4934,5156,4496,True
dev,fastapi_15588,6,88.25,88.26,11717,996,read_file,49,1980,3534,1708,False
dev,fastapi_15588,7,135.48,135.49,12433,1616,write_file,1230,58,4410,0,
dev,fastapi_15588,8,136.45,138.16,12827,21,run_command,35,2373,0,0,
dev,fastapi_15588,9,190.32,190.33,13586,1764,edit_file,633,613,5168,0,
dev,fastapi_15588,10,201.55,201.56,14048,370,edit_file,1530,206,0,0,
dev,fastapi_15588,11,202.82,202.83,14485,39,read_file,65,1410,0,1205,False
dev,fastapi_15588,12,237.32,237.33,3832,39,read_file,65,1348,0,1146,False
dev,fastapi_15589,1,6.03,6.22,5297,143,run_command,48,7380,510,0,
dev,fastapi_15589,2,15.95,15.96,9211,245,read_file,45,4941,809,4605,True
dev,fastapi_15589,3,18.23,18.24,10634,42,read_file,81,6521,0,6203,True
dev,fastapi_15589,4,20.90,20.91,12510,43,read_file,82,5789,0,5463,True
dev,fastapi_15589,5,64.25,64.26,14275,1435,read_file,33,5362,4975,5035,True
dev,fastapi_15589,6,66.55,66.56,15813,36,read_file,69,5526,0,5217,True
dev,fastapi_15589,7,98.68,98.69,7233,286,read_file,69,5639,955,5330,True
dev,fastapi_15589,8,100.87,100.88,8829,36,read_file,69,5477,0,5174,True
dev,fastapi_15589,9,190.69,190.70,10384,3088,write_file,1784,64,9101,0,
dev,fastapi_15589,10,191.75,192.54,10921,21,run_command,41,228,0,0,
dev,fastapi_15589,11,231.26,231.27,11032,1341,edit_file,766,588,3991,0,
dev,fastapi_15589,12,232.25,232.94,11464,21,run_command,41,282,0,0,
dev,fastapi_15661,1,15.33,15.35,5260,257,run_command,53,98,877,0,
dev,fastapi_15661,2,19.06,19.07,5341,132,read_file,50,1249,359,1022,False
dev,fastapi_15661,3,28.89,28.90,5837,342,run_command,26,393,1277,0,
dev,fastapi_15661,4,39.71,39.78,6033,386,run_command,64,2025,1281,0,
dev,fastapi_15661,5,51.47,51.48,6746,404,read_file,30,4910,1378,4418,True
dev,fastapi_15661,6,62.79,62.79,8772,349,read_file,31,4096,1194,3666,True
dev,fastapi_15661,7,73.92,73.95,10373,350,run_command,128,1053,1118,0,
dev,fastapi_15661,8,81.05,81.06,10819,236,read_file,45,1311,829,1114,False
dev,fastapi_15661,9,99.63,99.64,11362,632,run_command,43,2698,2101,0,
dev,fastapi_15661,10,114.97,114.98,12239,507,read_file,35,1253,1798,1080,False
dev,fastapi_15661,11,136.56,136.58,12597,731,read_file,45,10848,2569,10000,True
dev,httpx_3672,1,8.84,8.85,3371,298,read_file,37,4614,936,4272,True
dev,httpx_3672,2,13.79,13.80,4954,140,read_file,56,6039,382,5646,True
dev,httpx_3672,3,16.00,16.01,6842,30,read_file,56,5564,0,5224,True
dev,httpx_3672,4,27.77,27.79,8579,368,read_file,36,4281,1199,3986,False
dev,httpx_3672,5,67.64,67.65,9891,1364,read_file,37,3752,4795,3477,False
dev,httpx_3672,6,98.64,98.65,11051,1052,read_file,37,4222,3747,3918,True
dev,httpx_3672,7,152.85,152.86,12407,1838,read_file,37,4340,6797,4007,True
dev,httpx_3672,8,154.81,154.82,13776,30,read_file,56,3298,0,3039,False
dev,httpx_3672,9,235.77,235.78,14798,2717,run_command,43,373,9756,0,
dev,requests_7505,1,7.38,7.39,3703,235,run_command,52,1025,870,0,
dev,requests_7505,2,11.32,11.32,4062,136,read_file,38,3752,418,3439,True
dev,requests_7505,3,18.65,18.66,5308,237,run_command,79,282,722,0,
dev,requests_7505,4,24.95,24.96,5448,225,read_file,74,562,600,396,False
dev,requests_7505,5,26.18,26.19,5687,37,read_file,74,612,0,451,False
dev,requests_7505,6,27.39,27.40,5911,37,read_file,74,620,0,459,False
dev,requests_7505,7,36.95,36.96,6139,340,read_file,38,5213,1156,4889,True
dev,requests_7505,8,97.25,97.27,7733,2097,run_command,53,803,7804,0,
dev,requests_7505,9,98.13,98.14,8042,21,read_file,37,4620,0,4264,True
dev,requests_7505,10,100.12,100.13,9453,33,run_command,79,150,0,0,
dev,requests_7505,11,101.30,101.31,9559,37,read_file,73,426,0,262,False
dev,requests_7505,12,102.56,102.57,9775,37,read_file,73,701,0,525,False
dev,requests_7505,13,184.41,184.42,10069,2842,edit_file,191,529,10018,0,
dev,requests_7505,14,186.96,186.97,10323,79,edit_file,275,679,0,0,
dev,requests_7505,15,188.98,188.99,10611,63,edit_file,172,459,0,0,
dev,requests_7505,16,191.02,191.04,10860,63,edit_file,184,520,0,0,
dev,rich_4070,1,17.21,17.22,4721,570,read_file,31,7462,1745,7135,True
dev,rich_4070,2,23.62,23.63,6772,178,read_file,50,5716,525,5319,False
dev,rich_4070,3,83.21,83.22,8591,2050,edit_file,1188,1160,6360,0,
dev,rich_4070,4,95.70,95.71,9237,422,edit_file,1613,493,0,0,
dev,rich_4070,5,100.65,100.66,9833,156,edit_file,532,568,0,0,
dev,rich_4070,6,101.40,101.41,10186,19,read_file,31,4459,0,4039,True
dev,rich_4070,7,103.38,103.39,11750,27,read_file,50,5184,0,4791,True
dev,rich_4070,8,105.40,105.41,13370,27,read_file,50,5159,0,4809,True
dev,rich_4070,9,107.43,107.44,14941,27,read_file,50,5744,0,5361,True
dev,rich_4070,10,147.87,147.89,7458,359,read_file,31,7295,1051,6968,True
dev,rich_4070,11,160.14,160.16,9478,378,read_file,50,5882,1103,5489,True
dev,rich_4070,12,177.51,177.53,11314,555,read_file,48,4459,1861,4039,True
dev,rich_4070,13,195.96,195.98,12884,595,run_command,50,180,1947,0,
dev,rich_4070,14,196.90,196.93,12992,26,run_command,55,438,0,0,
dev,rich_4070,15,197.92,197.95,13159,27,run_command,52,108,0,0,
dev,rich_4070,16,229.75,229.79,13250,1094,run_command,243,1260,3252,0,
dev,rich_4077,1,4.53,4.59,3758,127,run_command,35,1612,334,0,
dev,rich_4077,2,6.97,6.98,4359,72,read_file,31,4460,163,4033,True
dev,rich_4077,3,8.93,8.94,5933,31,run_command,49,308,0,0,
dev,rich_4077,4,10.06,10.06,6082,35,read_file,67,945,0,772,False
dev,rich_4077,5,21.15,21.16,6421,393,run_command,56,238,1253,0,
dev,rich_4077,6,22.27,22.28,6547,35,read_file,67,4739,0,4492,False
dev,rich_4077,7,24.18,24.19,7846,35,read_file,67,3960,0,3697,False
dev,rich_4077,8,26.01,26.02,8996,35,read_file,67,1220,0,1030,False
dev,rich_4077,9,27.14,27.15,9418,28,run_command,58,100,0,0,
dev,rich_4077,10,28.24,28.25,9503,35,read_file,67,1266,0,1076,False
dev,rich_4077,11,36.52,36.53,9920,280,run_command,51,148,847,0,
dev,rich_4077,12,37.66,37.67,10026,35,read_file,67,596,0,427,False
dev,rich_4077,13,81.13,81.14,10295,1503,run_command,49,285,5009,0,
dev,rich_4077,14,119.50,119.51,10434,1327,read_file,67,563,4413,401,False
dev,rich_4077,15,151.49,151.51,10693,1106,read_file,32,5428,3645,5091,True
dev,rich_4077,16,153.55,153.56,12274,28,run_command,54,95,0,0,
dev,rich_4077,17,154.70,154.71,12359,35,read_file,68,387,0,234,False
dev,rich_4077,18,155.65,155.66,12551,26,run_command,58,167,0,0,
dev,rich_4077,19,156.87,156.88,12657,37,read_file,70,4147,0,3889,False
dev,rich_4077,20,158.58,158.59,13809,27,run_command,62,241,0,0,
dev,rich_4077,21,159.72,159.74,13933,34,run_command,66,208,0,0,
dev,rich_4077,22,160.91,160.93,14059,35,read_file,68,990,0,704,False
dev,rich_4077,23,162.06,162.08,14439,27,run_command,62,240,0,0,
dev,rich_4077,24,200.69,200.71,3104,338,run_command,91,239,955,0,
dev,rich_4077,25,203.59,203.60,3244,103,read_file,31,4573,270,4033,True
dev,rich_4077,26,208.71,208.73,4853,147,run_command,74,240,414,0,
dev,rich_4077,27,209.63,209.65,4981,28,run_command,58,213,0,0,
dev,rich_4077,28,210.72,210.74,5101,35,read_file,67,973,0,678,False
dev,rich_4077,29,214.21,214.22,5460,117,read_file,67,621,285,353,False
dev,rich_4077,30,217.39,217.40,5705,107,read_file,67,924,256,636,False
dev,rich_4077,31,218.06,218.09,6052,13,submit_patch,2,53,0,0,
dev,rich_4077,32,229.46,229.46,6100,412,,0,0,0,0,
base,fastapi_15588,1,6.16,7.33,5441,96,search_similar_code,31,74,301,0,
base,fastapi_15588,2,9.38,9.41,5503,51,search_similar_code,16,1266,112,0,
base,fastapi_15588,3,10.87,11.07,6007,26,run_command,40,5891,0,0,
base,fastapi_15588,4,16.76,16.77,8377,97,read_file,30,5079,284,4684,True
base,fastapi_15588,5,56.90,56.91,9943,962,read_file,49,1980,3427,1708,False
base,fastapi_15661,1,19.41,19.43,5259,287,run_command,41,131,1104,0,
base,fastapi_15661,2,25.23,25.24,5349,144,read_file,50,1249,404,1022,False
base,fastapi_15661,3,36.85,36.86,5845,286,run_command,26,393,1007,0,
base,fastapi_15661,4,55.75,55.82,6041,479,run_command,64,2025,1806,0,
"""
TASKS_CSV = """task,repo,wall_s,tool_calls,patch_bytes,resolved,test_exit_code,outcome,first_event_s,last_event_s
fastapi_15661,fastapi/fastapi,143.4,11,0,False,-1,refused (context window),6.18,142.74
fastapi_15588,fastapi/fastapi,250.0,12,1884,False,1,time budget (4 min),5.13,242.44
rich_4070,Textualize/rich,247.2,16,1847,False,1,time budget (4 min),1.88,231.64
requests_7505,psf/requests,316.0,16,1818,False,1,time budget (4 min),0.77,191.8
httpx_3672,encode/httpx,240.6,9,0,False,-1,time budget (4 min),0.18,235.95
rich_4077,Textualize/rich,234.3,30,0,False,1,tool budget (30 calls),2.02,231.48
fastapi_15589,fastapi/fastapi,249.5,12,1912,False,1,time budget (4 min),5.43,237.68
"""
GPU_GZ_B64 = "H4sIAAAAAAAC/5XdUYojxxJG4fe7ilmAsSuiqiSF97+w25GViWro1vicJ4P5MCfkgf4ZGmVu+fhnq3/27dd2/Luf/2b9HRl//dr++pXbY6uvf+zbfnz9i//lj3b/61dQ+/gS1L7++rUDux9ftljvsiFsQrtvsLcuuwkbwn7rfdZn/C34cX7HR1/305+IeH7G35Jj2z5r8iEvSz7koz4lb5904uapU+kd6fO49KZ0KJ1Kw+6+csfdU4fSqTTpfvy7Hbz7rYPrg3W/NeyuS29Kh9KpNOuOvvKk3UuH0qk07C7VXaq7VHeZ7uwrH7R76VA6lYbdpbpbP3H31Kk06x5D4Em7lw6lU2nY3Ve+cPfUoXQqzbrHJijavXQonUrD7lLdX/rYcPfUqTTr7lVwbLR76eA6aPfSsLsuvSkdSqfSpPs5VsEB9+Bbh9KpNOzuK3fcPXUonUqz7l4FvHvog3YvnUrD7rr0pnQonUqz7l4Fx0m7lw6lU2nYXaq79QN3T51Ks+5eBceDdi8dSqfSsLuvfOLuqUPpVJp19yrg3UuH0qk07O4rX7h76lA6lWbdYxWgvwx82xA2hd2JrQ+9P19XH4r/qBPrE+7A11gD50878FWf8X9/0G/7Lfr1+ox38h/u++ACfOtQOpVmn3TvAN69dHCdtHtp2F2X3pQOpVNp1t074Nxp99KhdCoNu0t1tz5w99SpNOvuHXAetHvpUDqVht195Ym7pw6lU2nW3TvgfNDupUPpVBp2l+ou1V2qu/WTdvcaOJ+0e+lQOpWG3X3lC3dPHUqn0qS7rk0Au6eGi+qtU2nYXZfelA6lU2nW3avgsdHupUPpVBp2l+puHbh76lSadfcqeATtXjqUTqVhd1+ZuHvqUDqVZt29Cnj30qF0Kg27+8odd08dSqfSrLtXweOg3UuH0qk07C7VXaq7VHeZ7l4Fj5N2Lx1Kp9Kwu6984O6pQ+lUGnQf21gFD7YHbzqUTqVhd6nuUt2lust0j1Xwot1Lh9KpNOzuKwt3Tx1Kp9Kse6wC3L10KJ1Kw+6vK58b7p46lE6lWXevAt49dNDupVNp2F2X3pQOpVNp1t2r4Jm0e+lQOpWG3aW6W++4e+pUmnX3KnjutHvpUDqVht195YG7pw6lU2nSHWMV0O63DqVTadjdV564e+pQOpVm3b0KePfQD9q9dCoNu+vSm9KhdCrNunsVPJ+0e+lQOpWG3aW6S3WX6i7TPVbBi3YvHUqn0rC7VHfrwt1Tp9Kse6yCot1Lh9KpNOz+uvK14e6pQ+lUmnX3KuDdQwftXjqVht116f/8TZmbDWFT2P/8/Y0jxxp4Jel92xA2hUW99aH3h/8bU8Pd+tap9I50r4DXTruXDqVTadjdVx64e+pQOpVm3b0CePfQJ+1eOpWG3XXpTelQOrl+0O5eAa8H7V46lE6lYXdf+cTdU4fSqTTr7hXAu5cOrl+0e2nY3VcW7p46lE6lWXevgNpo99KhdCoNu0t1l+ou1V28ex+boODfY751KJ1Kw+5S3a0Td0+dSrPuXgWVtHvpUDqVht195Y67pw6lU2nW3augDtq9dCidSsPuUt2lukt1l+nuVVAn7V46lE6lYXdf+cDdU4fSqTTr7lXAu4d+0u6lU2nYXZfelA6lU2nWPVbBi3YvHUqn0rC7ryzcPXUonUqT7uNaBbD7rUPpVBp219+5bbh76lA6lWbdX6sgt6DdS4fSqTTsLtVdqrtUd5nu7CuTdi8dSqfSsLuv3HH31KF0Ks2698N0Lx1Kp9Kwu688cPfUoXQqzbqPw3QPfdLupVNp2F2X3pQOpVNp1n32lQ/avXQonUrD7r7yibunDqVTadJ99irA3W8dSifXL9xdl96UDqVTadY9VkHR7qVD6VQadpfqLtVdqrtMd6+C2Gj30qF0Kg27S3W3Dtw9dSrNunsVRNDupUPpVBp295WJu6cOpVNp1t2rgHcPvdPupVNp2F2X3pQOpVNp1t2rIA7avXQonUrD7lLdrU/cPXUqTbrH97IOvSkdSqfSsLuvfJDfl3nbEDaF3YHtNRAP+jkP/aSf89KpNPucew3EE3eX6i7VXaZ7rIEX7V46lE6lYXep7taFu6dOpVn3WANFu5cOpVNp2P11ZW64e+pQOpVm3b0GePfQQbuXTqVhd116UzqUTqVZd6+BTNq9dCidSsPuUt2td9w9dSpNusf3sQ69KR1Kp9Kwu688cPfUoXQqzbp7FfDuoU/avXQqDbvr0pvSoXQqzbp7FeSDdi8dSqfSsLuvfOLuqUPpVJp19yrg3UuH0qk07O4rX7h76lA6lWbdYxXg7qGLdi+dSsPuuvS37ucf8Lfs1/4Zf3/AKD7j7w8YPb7jXgT7Rj/rpUPpVJp91r0I4CtGNx1Kp9Kke3wva8JXjG46lE6lYXdfuePuqUPpVJp19yLg3UuH0qk07O4rD9w9dSidSrPuXgTwNaCbDqVTadhdqrtUd6nuMt29COBrQDcdSqfSsLtUd+sn7p46lWbdvQjgK0Y3HUqn0rC7r3zh7qlD6VSadY9VgLuHLtq9dCoNu+vSm9KhdCpNusc3syZ8xeimQ+lUGnZ/XQlfMbrpUDqVZt29Cnj30qF0Kg27+8rE3VOH0qk06+5VwLuXDq532r007K5Lb0qH0qk06+5VAF9fuulQOpWG3aW6W5+4e+pUmnX3KoCvL910KJ1Kw+6+8oG7pw6lU2nW3auAdw/9pN1Lp9Kwuy69KR1Kp9Kg+9yuVcD24E2H0qk07C7V3bpw99SpNOseq4C8GnWzIWwKuxP7dd25wd5pQ9gUlvT2CqC9y4awKSzq7dsC9k4bwqawpLd/6p/JepcNYVNY1Fuit0Rv2x32Tkt6+6f8j28r/cGGsCks6u3bDtg7bQibwpLe/qn+41tKf7AhbAqLekv0lugt0Vu0d3yTap7kd0FvNoRNYVFv3/aEvdOGsCks6e2f3bR32RA2hUW9fdsL9k4bwqawpHf87C7Wu2wIm8Ki3vrQ+8P6fOvAGr6JdNM70v0zHL6JdNOhdCoNu/vKwN1Th9KpNOvun+m8e+ngOmn30rC7Lr0pHUqn0qy7f8bDN5FuOpROpb91748fcKns1gfOnjqVJh/3+LrUhE8i3XQonUrD7r7yxN1Th9KpNOvuCcC7h37Q7qVTadhdl96UDqVTadbdkwA+iXTToXQqDbtLdZfqLtVdpnuMgp9m61Gf8bfs8/kZf6s+4zOG0fUh+rMu+mEvnUqz7rEIinYvHUqn0rD760r4HtJNh9KpNOvuRcC7hw7avXQqDbvr0pvSoXQqTbrHt6UmfA/ppkPpVBp2l+ou1d16x91Ts+6eBPAdp5sOpVNp2N1XHrh76lA6lWbdPQl499KhdHJ94u669KZ0KJ1Ks+5eBfA9pJsOpVNp2F2qu1R3qe4y3b0K4HtINx1Kp9Kwu1R36xfunjqVZt1jFbxo99KhdCoNu/vKwt1Th9KpNOk+rlUAuy8N30O66VQadtelN6VD6VSadfcqeAXtXjqUTqVhd6nuUt2lulsn7e5VAN9HuulQOpWG3X3ljrunDqVTadbdqwC+M3TToXQqDbtLdZfqLtVdprtXAXwf6aZD6VQadveVD9w9dSidSrPuXgW8e+lQOpWG3X3lE3dPHUqn0qT7+rbUF9yDbx1Kp9Kwu1R368LdU6fSrHusgqLdS4fSqTTs/roSvut006F0Ks26exXw7qGDdi+dSsPuuvSmdCidSrPuXgXwXaebDqVTadhdqrv1jrunTqVZd68C+B7VTYfSqTTs7isP3D11KJ1Ks+5eBbx76JN2L51Kw+669KZ0KJ1Kk+7r21Lhu043HUqn0rC7VHep7lLdZbp7FcD3qG46lE6lYXep7tYv3D11Ks26xyp40e6lQ+lUGnb3lYW7pw6lU2nW/bUKdvge1U2H0qk07C7VXaq7deDuqVn3cVx6UzqUTqVhd1+ZuHvqUDqVZt1nX7nT7qVD6VQadveVB+6eOpROpUn3+NbUHb6PdNOhdCoNu0t1l+ou1V2mO/rKB+1eOpROpWF3qe7WT9w9dSrNuvO49KZ0KJ1Kw+6+8oW7pw6lU2nWPVYB7l46lE6lYXdfWbh76lA6lWbdvQrg+0g3HUqn0rC7VHep7lLdZbp7FcB3nW46lE6lYXep7taJu6dOpUn3+ObUHb6PdNOhdCoNu/vKA3dPHUqn0qy7VwHvXjqUTqVhd1954u6pQ+lUmnX3KuDdQz9o99KpNOyuS29Kh9KpNOvuVQDfSbrpUDqVht2lukt1t37h7qlZ91gFL9q9dCidSsPuvrJw99ShdCrNuscqwN2t4TtJN51Kw+669KZ0KJ1Kk+7xzak7fCfppkPpVBp2l+punbh76lSadfcqgO8k3XQonUrD7r4SfXfT24awKewObK8B2jvswXqXTWFRb12W/blYOpROpdmfi14B8F2kmw6lU2nYXaq7VHfrB+6emnX3CoDvOd10KJ1Kw+6+8om7pw6lU2nWPVYA7l46lE6lYXdf+cLdU4fSqfR/dp//btu1Akj3W5Pd+ptOpWF3XXpTOpROpVl3rwD0ptNvOpROpWF3qe7WgbunTqVZd28C9BbVbzqUTqVhd1+ZuHvqUDqVZt29Cnj30DvtXjqVht116U3pUDqVZt29CtBbVL/pUDqVht195Ym7pw6lU2nW3auAdy8dSqfSsLuvfODuqUPpVJp0j+9f3dGbTr/pUDqVht2lukt1l+ou0z1WwYt2Lx1Kp9Kwu1R368LdU6fSrHusgqLdS4fSqTTs/rry2HD31KF0Ks26exXw7qGDdi+dSsPuuvSmdCidSrPuXgXHTruXDqVTadhdqrtUd+sDd0/NunsVHAftXjqUTqVhd1954u6pQ+lUmnSPr2jF3W8dSifXD9xdl96UDqVTadbdq+B40u6lQ+lUGnaX6i7VXaq7THevguNFu5cOpVNp2N1XFu6eOpROpVn3WAW4e+lQOpWG3V9XnhvunjqUTqVZd6+CM2j30qF0Kg27S3WX6i7VXaa7V8GZtHvpUDqVht2lulvvuHvqVJp0j29p3U+4Y986lE6lYXdfeeDuqUPpVJp19yo4T9q9dCidSsPu+tD9BxvCprA7sL0Gzgf9nJcOpVNp9jn3GgBvWv1mg9on+5yX/eFz/j9rMrjrV/EAAA=="   # nvidia-smi every 5 s during the 7 tasks: timestamp, gpu, MiB used, MiB total, util %
SHA256 = {"calls": "0f297d3e71de0417f77aa4f0274836964b2bdf4e6878e3bbaab1e5e6189458ed", "tasks": "fa16ab0e8d6368ab8a2d2bbc1727c55a66fc32c9a6e1437cdee78d9ab09c0dd2", "gpu": "3b3693a6680b838307fa23143decdf64fd1e0ca6a903bac72c9b09d762da1710"}

gpu_raw = gzip.decompress(base64.b64decode(GPU_GZ_B64))
for name, blob in (("calls", CALLS_CSV.encode()), ("tasks", TASKS_CSV.encode()), ("gpu", gpu_raw)):
    assert hashlib.sha256(blob).hexdigest() == SHA256[name], f"{name} table does not match its checksum"
calls = pd.read_csv(io.StringIO(CALLS_CSV), keep_default_na=False)
tasks = pd.read_csv(io.StringIO(TASKS_CSV)).set_index("task")
gpu = pd.read_csv(io.BytesIO(gpu_raw), header=None, skipinitialspace=True,
                  names=["ts", "gpu", "mem_used_mib", "mem_total_mib", "util_pct"])
dev = calls[calls.run == "dev"].sort_values(["task", "call"]).reset_index(drop=True)
base = calls[calls.run == "base"].sort_values(["task", "call"]).reset_index(drop=True)
dev["next_prompt"] = dev.groupby("task").prompt_tokens.shift(-1)
for d_ in (dev, base):   # model_s: from the previous tool result to this call's answer; tool_s: running the tool
    d_["model_s"] = d_.t_s - d_.groupby("task").tool_done_s.shift(1).fillna(0.0)
    d_["tool_s"] = d_.tool_done_s - d_.t_s

RUN = {"date": "2026-09-30", "hardware": "4 x L4, vLLM tensor parallel 4", "order": "one task at a time",
       "evaluator": "the official starter's local evaluator (subprocess sandbox)"}
EVAL_CONFIG = {"host sample": {"timeout_seconds": 60, "max_tool_calls": 10, "max_time_minutes": 1, "max_turns": 50},
               "this run": {"timeout_seconds": 120, "max_tool_calls": 30, "max_time_minutes": 4, "max_turns": 60}}
print(f"run: {RUN['date']}, {RUN['hardware']}, {RUN['order']}, {RUN['evaluator']}")
for k_, v_ in EVAL_CONFIG.items():
    print(f"eval_config.yaml, {k_}: " + ", ".join(f"{a} {b}" for a, b in v_.items()))
print("other change: the sample's 2 placeholder LoRA adapters removed (the adapter: lines and adapters/); "
      "prompts and configs/sampling.yaml are checked by SHA-256 in section 1")
print(f"model calls stored: {len(calls)} ({len(dev)} from the 7-task run, {len(base)} from the starter's 2-task run)")
print(f"tasks stored: {len(tasks)}; GPU samples: {len(gpu):,}; all three tables match their SHA-256")
print(f"knobs: MAX_OUTPUT_TOKENS={MAX_OUTPUT_TOKENS:,} THINKING_BUDGET={THINKING_BUDGET:,} MAX_TURNS={MAX_TURNS} "
      f"MAX_TIME_MINUTES={MAX_TIME_MINUTES} N_HIDDEN_TASKS={N_HIDDEN_TASKS}")

## 1. The trap, in the host's own files

Three lines decide it. The sample asks for 16,384 output tokens, the server's context is 32,768, and the README's
best-practices section uses the same pairing as its example (`thinking_budget` 4096 with `max_output_tokens: 16384`).
It also lists 16,384 as the default, so, per the README, omitting the key does not help. The harness does protect long
trajectories with compaction, and its trigger, 14,336, is exactly the 16,384 ceiling minus 2,048. Section 3 shows why
2,048 is not enough.

In [2]:
# CHECKED (when the competition data is attached): the host's lines and files below, against the mounted copies.
# COPIED (cannot be re-checked from here): my run-log lines, the official starter's lines, page quotes, my public
# score, and figures reported in other notebooks. Each copied item is labelled where it is printed.
HOST = {
    "sample_submission/configs/sampling.yaml": [
        "max_output_tokens: 16384",
        "  thinking_budget: 4096",
        "  include_thoughts: true",
    ],
    "HARNESS_README.md": [
        "| | `max_output_tokens` | `1` to `32,768` (default `16,384`; bounded by 4x L4 vLLM `max_model_len=32768`) |",
        "  - `max_model_len = 32768` (`32,768` tokens maximum combined prompt + reasoning + output context length)",
        "  - `gpu_memory_utilization = 0.80` (`~19.2 GB` usable per GPU / `~76.8 GB` usable across 4 GPUs)",
        '  - `default_chat_template_kwargs = {"enable_thinking": True}`',
        "  - `compaction_interval = 5`",
        "  - `token_threshold = 14,336`",
        "## 10. Competitor Best Practices & Important Gotchas",
        "Set a reasonable `thinking_budget` (e.g., `4096` with `max_output_tokens: 16384`) and instruct your agent to apply incremental edits.",
        "- **Output Truncation**: Both `stdout` and `stderr` are truncated to `max_stdout_chars` (**`5,000` characters** default).",
        "- **Dual Truncation Cap**: Output is capped at **`150` lines** (`max_file_lines`) AND **`10,000` characters** (`max_file_chars`).",
        "- **Docker Backend (`--sandbox docker`, default)**:",
    ],
}
# SHA-256 of the files this run used, and the run's own check against the mounted host sample (copied from my run log)
RUN_FILES_SHA256 = {
    "sample_submission/configs/sampling.yaml": "3dab0b2506dae34ce92fef7b380d6073729104b23fe2cf66d44ac1d2f213aa6a",
    "sample_submission/prompts/system.md": "f1cbf7943ee9687382321b1dfd9cd88581d608c1c8b8e6ea61a763bd1a3234af",
    "sample_submission/prompts/analyzer.md": "c762b062032cc8463015e589ad8b4f0e17296062e1b0ff1e78712eef9f5e2d43",
}
RUN_LOG_SHA_CHECK = [   # [64.96 s] in my run log: status, file, host sha256 prefix, run sha256 prefix
    "identical  configs/sampling.yaml                         3dab0b2506da   3dab0b2506da",
    "identical  prompts/analyzer.md                           c762b062032c   c762b062032c",
    "identical  prompts/system.md                             f1cbf7943ee9   f1cbf7943ee9",
    "changed    eval_config.yaml                              adb486b67535   dc82f61ec847",
]
STARTER = {   # COPIED: Ryan Holbrook's "Getting Started - Gemma 4 Developer Agent", as run on 2026-09-30
    "cell 4 (rewrites sampling.yaml)": "    'max_output_tokens: 16384\\n'",
    "cell 8 (vLLM server)": "    gpu_memory_utilization=0.90,",
    "cell 10 (evaluator)": "        compaction_interval=15,",
    "cell 10 (evaluator) ": "        token_threshold=14336,",
}
REFUSAL = ("This model's maximum context length is 32768 tokens. However, you requested 16384 output tokens and "
           "your prompt contains at least 16385 input tokens, for a total of at least 32769 tokens. Please reduce "
           "the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=16385)")

DATA = INPUT / "competitions" / "gemma-4-developer-agent"


def mounted(rel):
    p = DATA / rel
    if p.is_file():
        return p
    hits = [q for q in INPUT.rglob(Path(rel).name) if q.is_file() and str(q).endswith(rel)] if INPUT.exists() else []
    hits.sort(key=lambda q: ("paper" in str(q), len(str(q))))    # the main competition's copy first
    return hits[0] if hits else None


for rel, lines in HOST.items():
    p = mounted(rel)
    text = p.read_text(encoding="utf-8", errors="replace") if p else None
    print(f"{rel}: {'CHECKED line by line against the mounted file' if p else 'not mounted: stored quotes only'}")
    for line in lines:
        status = "" if text is None else ("present  " if line in text else "NOT FOUND")
        print(f"  {status} {line.strip()}")
print("\nfiles this run used vs the mounted host sample (SHA-256):")
for rel, sha in RUN_FILES_SHA256.items():
    p = mounted(rel)
    got = hashlib.sha256(p.read_bytes()).hexdigest() if p else None
    status = "not mounted" if got is None else ("identical" if got == sha else "DIFFERENT")
    print(f"  {status:<11} {rel:<40} run {sha[:12]}  mounted {got[:12] if got else '-'}")
print("the run's own check at the time (COPIED from my run log):")
for line in RUN_LOG_SHA_CHECK:
    print("  " + line)
print("\nofficial starter (COPIED):")
for where, line in STARTER.items():
    print(f"  {where.strip():<32} {line.strip()}")

SAMPLE_MAX_OUT = int(re.search(r"max_output_tokens: (\d+)", HOST["sample_submission/configs/sampling.yaml"][0]).group(1))
MAX_MODEL_LEN = int(re.search(r"max_model_len = (\d+)", HOST["HARNESS_README.md"][1]).group(1))
TRIGGER = int(re.search(r"token_threshold = ([\d,]+)", HOST["HARNESS_README.md"][5]).group(1).replace(",", ""))
assert MAX_MODEL_LEN == CONTEXT and TRIGGER == COMPACTION_TRIGGER
req_out, min_in = map(int, re.search(r"requested (\d+) output tokens and your prompt contains at least (\d+) input",
                                     REFUSAL).groups())
CEIL_SAMPLE = rec("ceiling_sample", CONTEXT - SAMPLE_MAX_OUT)
MARGIN_SAMPLE = rec("margin_sample", CEIL_SAMPLE - COMPACTION_TRIGGER)
print(f"\nvLLM refusal in my run (COPIED from the log): \"{REFUSAL[:160]}...\"")
print(f"sample cap {n(SAMPLE_MAX_OUT)} + context {n(CONTEXT)}: prompt ceiling = {n(CONTEXT)} - {n(SAMPLE_MAX_OUT)} = "
      f"{n(CEIL_SAMPLE)} tokens; a prompt of {n(min_in)} asks for {n(min_in + req_out)} > {n(CONTEXT)} and is refused")
print(f"compaction trigger {n(COMPACTION_TRIGGER)}: {n(CEIL_SAMPLE)} - {n(COMPACTION_TRIGGER)} = {n(MARGIN_SAMPLE)} tokens "
      f"between the point where history gets compacted and the point where the call is refused")
print(f"with a cap of 8,192: ceiling {n(CONTEXT - 8_192)}, room between trigger and ceiling {n(CONTEXT - 8_192 - COMPACTION_TRIGGER)}")

## 2. What happened on 7 development tasks

Seven public development tasks, chosen before the run, one task at a time. `peak_prompt` is the largest prompt a call
was answered with, `compactions` counts history compactions, and `output_tokens` is everything the traced agent calls
generated (thinking included; compaction summaries not counted).

Five tasks ran out of time, one spent its 30 tool calls, and `fastapi_15661` was refused by vLLM at its 12th call after
a 10,848-character `read_file` reply, before any edit: it produced no patch. None of the 7 resolved in 4 minutes, so a
refusal costs 1/58 of the public score (0.0172) only on a task that would otherwise be solved
([why 58](https://www.kaggle.com/code/dariushafshar/gemma-4-lb-58-tasks-bronze-is-a-124-team-tie)). This exact config
scored 0.06 on the public board, which is 4 of 58 tasks (Kaggle truncates to two decimals; 4/58 = 0.069 is the only
count that displays as 0.06).

In [3]:
dev["compacted_next"] = dev.next_prompt.notna() & (dev.next_prompt < dev.prompt_tokens)
per = dev.groupby("task").agg(model_calls=("call", "max"), first_prompt=("prompt_tokens", "first"),
                               peak_prompt=("prompt_tokens", "max"), output_tokens=("completion_tokens", "sum"),
                               prompt_tokens_total=("prompt_tokens", "sum"), compactions=("compacted_next", "sum"))
T = tasks.join(per)
cols = ["repo", "outcome", "wall_s", "model_calls", "tool_calls", "peak_prompt", "compactions", "output_tokens", "patch_bytes"]
print(T[cols].to_string())

dead = T[T.outcome.str.startswith("refused")]
k, N = len(dead), len(T)
lo, hi = beta.ppf(0.025, k, N - k + 1), beta.ppf(0.975, k + 1, N - k)
DEAD = dead.index[0]
dcalls = dev[dev.task == DEAD]
last = dcalls.iloc[-1]
edits = int(dcalls.tool.isin(["edit_file", "write_file"]).sum())
print(f"\nrefused: {rec('dead_k', k)} of {rec('dead_n', N)} traced tasks (exact 95% interval {rec('ci_lo', f'{100 * lo:.1f}')}% "
      f"to {rec('ci_hi', f'{100 * hi:.1f}')}%)")
print(f"{DEAD}: {len(dcalls)} calls answered, call {len(dcalls) + 1} refused at {T.loc[DEAD, 'wall_s']} s; "
      f"edits made before it: {edits}; patch bytes: {T.loc[DEAD, 'patch_bytes']}")
print(f"  last answered prompt {n(last.prompt_tokens)} tokens; the {last.tool} reply that followed it: "
      f"{n(last.obs_chars)} characters")
others = T.drop(index=DEAD)
close = others.peak_prompt.idxmax()
print(f"closest survivor: {close} peaked at {n(others.peak_prompt.max())} tokens, "
      f"{n(CEIL_SAMPLE - others.peak_prompt.max())} below the {n(CEIL_SAMPLE)} ceiling")
print(f"tasks that ended on the 4-minute budget: {int((T.outcome == 'time budget (4 min)').sum())}; "
      f"on the 30-call budget: {int((T.outcome == 'tool budget (30 calls)').sum())}; resolved: {int(T.resolved.sum())} of {N}")
print(f"peak prompt per task: {n(T.peak_prompt.min())} to {n(T.peak_prompt.max())}; model calls per task: "
      f"{int(T.model_calls.min())} to {int(T.model_calls.max())}")

PUBLIC_TASKS = 58     # public split size (see the linked leaderboard notebook)
TRUNC_DECIMALS = 2    # Kaggle truncates this board to 2 decimals (Meta Kaggle ScoreTruncationNumDecimals, same notebook)
V1_PUBLIC = 0.06      # COPIED: this exact config's public score, submitted 2026-09-30
shown = lambda k_: math.floor(k_ / PUBLIC_TASKS * 10 ** TRUNC_DECIMALS + 1e-9) / 10 ** TRUNC_DECIMALS
print(f"\none task on the {PUBLIC_TASKS}-task public split = {1 / PUBLIC_TASKS:.4f} of score")
for k_ in (3, 4, 5):
    print(f"  {k_}/{PUBLIC_TASKS} = {k_ / PUBLIC_TASKS:.4f} -> displayed {shown(k_):.2f}")
match = [k_ for k_ in range(PUBLIC_TASKS + 1) if abs(shown(k_) - V1_PUBLIC) < 1e-9]
print(f"this config's public score {V1_PUBLIC:.2f}: task counts that display as {V1_PUBLIC:.2f} under truncation: {match} "
      f"({match[0]}/{PUBLIC_TASKS} = {match[0] / PUBLIC_TASKS:.3f})")

In [4]:
fig, ax = plt.subplots(figsize=(11, 5.4))
others_pal = ["#2980b9", "#8e44ad", "#16a085", "#d68910", "#7f8c8d", "#2c3e50"]
j = 0
for task, g in dev.groupby("task"):
    if task == DEAD:
        style = dict(lw=2.6, color="#c0392b", zorder=5)
    else:
        style = dict(lw=1.4, color=others_pal[j % len(others_pal)], alpha=0.9)
        j += 1
    ax.plot(g.call, g.prompt_tokens, marker="o", ms=3.5, label=task, **style)
ax.plot([len(dcalls) + 1], [min_in], marker="^", ms=11, color="#c0392b", zorder=6)
ax.annotate(f"call {len(dcalls) + 1} refused:\nprompt at least {n(min_in)}", xy=(len(dcalls) + 1, min_in),
            xytext=(len(dcalls) + 4.5, min_in + 3_300), color="#c0392b", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#c0392b"))
for y, txt, st_ in ((COMPACTION_TRIGGER, f"{n(COMPACTION_TRIGGER)}: history is compacted after a call reaches this",
                     dict(color="#7f8c8d", ls=":")),
                    (CEIL_SAMPLE, f"{n(CEIL_SAMPLE)}: refused above this with max_output_tokens 16,384 (sample)",
                     dict(color="#c0392b", ls="--")),
                    (CONTEXT - 8_192, f"{n(CONTEXT - 8_192)}: refused above this with max_output_tokens 8,192",
                     dict(color="#27ae60", ls="--"))):
    ax.axhline(y, lw=1.2, **st_)
    ax.text(32.6, y + 250, txt, ha="right", va="bottom", fontsize=8.5, color=st_["color"])
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_xlim(0.5, 33)
ax.set_ylim(0, 27_000)
ax.set_xlabel("model call number within the task")
ax.set_ylabel("prompt tokens sent on that call")
ax.set_title("Prompt size per model call: the sample agent on 7 development tasks (4 x L4)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8.5)
plt.tight_layout()
plt.show()

Every task climbs by a few hundred to a few thousand tokens per call. Five of them crossed the grey line at 14,336, and
the four that made another call had their next prompt compacted. `fastapi_15661` never reached the grey line: it jumped
from 12,597 straight past the red one in a single step. The green line is where the same agent would be refused with
`max_output_tokens: 8192`.

## 3. Why compaction does not catch it

Compaction is the harness's defence against long trajectories, and it works: the traces show exactly when it fires. It
acts on the prompt that has just been sent (the next cell rules out prompt + output), so the next call is compacted only
after a call has already carried 14,336 or more. The most a single step can add safely is therefore the distance
between the trigger and the ceiling: 2,048 tokens with the sample cap, and more for a step that starts lower.

In [5]:
has_next = dev[dev.next_prompt.notna()]
above = has_next[has_next.prompt_tokens >= COMPACTION_TRIGGER]
below = has_next[has_next.prompt_tokens < COMPACTION_TRIGGER]
drops_above = int((above.next_prompt < above.prompt_tokens).sum())
drops_below = int((below.next_prompt < below.prompt_tokens).sum())
at_or_above = dev[dev.prompt_tokens >= COMPACTION_TRIGGER]
print(f"answered prompts at or above {n(COMPACTION_TRIGGER)}: {len(at_or_above)}; followed by another call: {len(above)} "
      f"(the other was {', '.join(sorted(set(at_or_above.task) - set(above.task)))}'s last call); next prompt smaller: {drops_above}")
print(f"prompts below {n(COMPACTION_TRIGGER)} followed by another call: {len(below)}; next prompt smaller: {drops_below}")
print(f"largest prompt with no drop after it: {n(below.prompt_tokens.max())}; smallest prompt with a drop after it: "
      f"{n(above.prompt_tokens.min())}")
# Prompt, or prompt + output? Calls whose prompt was under the trigger but whose prompt + output was not:
pt = below[(below.prompt_tokens + below.completion_tokens) >= COMPACTION_TRIGGER]
print(f"calls under the trigger by prompt but at or above it by prompt + output: {len(pt)}; next prompt smaller: "
      f"{int((pt.next_prompt < pt.prompt_tokens).sum())}")
for _, r in pt.iterrows():
    print(f"  {r.task} call {r.call}: prompt {n(r.prompt_tokens)} + output {n(r.completion_tokens)} = "
          f"{n(r.prompt_tokens + r.completion_tokens)}; next prompt {n(r.next_prompt)}")
print("  -> the trigger follows the prompt alone")

dev["prev_prompt"] = dev.groupby("task").prompt_tokens.shift(1)
dev["after_drop"] = dev.groupby("task").compacted_next.shift(1, fill_value=False).astype(bool)
after = dev[dev.after_drop]
long_calls = dev[dev.completion_tokens >= 300]
clean = long_calls[~long_calls.after_drop]
RATE = rec("rate", round(clean.completion_tokens.sum() / clean.model_s.sum(), 1))
RATE_ALL = round(long_calls.completion_tokens.sum() / long_calls.model_s.sum(), 1)
print(f"\noutput speed, prefill included: {RATE} tokens/s over the {len(clean)} calls with >= 300 output tokens that "
      f"did not follow a drop ({RATE_ALL} with the {len(long_calls) - len(clean)} that did)")
dev["extra_s"] = dev.model_s - dev.completion_tokens / RATE
after = dev[dev.after_drop]
normal = dev[~dev.after_drop & (dev.call > 1)]
ev = pd.DataFrame({"task": after.task, "call": after.call, "prompt_before": after.prev_prompt.astype(int),
                   "prompt_after": after.prompt_tokens, "seconds_beyond_output": after.extra_s.round(1)})
print("\n" + ev.to_string(index=False))
print(f"\nafter a drop, the call took {ev.seconds_beyond_output.min():.1f} to {ev.seconds_beyond_output.max():.1f} s more than "
      f"its output explains at {RATE} tokens/s; every other call: median {normal.extra_s.median():.2f} s, "
      f"max {normal.extra_s.max():.2f} s")
print(f"prompt right after a drop: {n(ev.prompt_after.min())} to {n(ev.prompt_after.max())} tokens "
      f"(median first prompt of a task: {n(per.first_prompt.median())})")

In [6]:
st = below.copy()
st["step"] = st.next_prompt - st.prompt_tokens
st["chars"] = st.obs_chars + st.args_chars
q = st.step.quantile([0.5, 0.9, 0.95])
print(f"steps (prompt growth from one call to the next, no drop in between): {len(st)}")
print(f"  median {n(q[0.5])}, 90th pct {n(q[0.9])}, 95th pct {n(q[0.95])}, largest {n(st.step.max())} tokens")
over = st[st.step > MARGIN_SAMPLE]
print(f"  steps larger than the {n(MARGIN_SAMPLE)}-token room: {len(over)} of {len(st)}, starting from prompts of "
      f"{', '.join(n(v) for v in sorted(over.prompt_tokens))}")
by_tool = st.groupby("tool").step.agg(["count", "median", "max"]).astype(int).sort_values("count", ascending=False)
print("\nsteps by tool:\n" + by_tool.to_string())
bump = st[(st.step >= 1_500) & (st.step < 2_100)]
print(f"steps of 1,500 to 2,100 tokens: {len(bump)}, by tool: {bump.tool.value_counts().to_dict()}")

FAIL_STEP_MIN = rec("fail_step_min", min_in - int(last.prompt_tokens))
fail_chars = int(last.obs_chars + last.args_chars)
print(f"\nthe refused step: from {n(last.prompt_tokens)} to at least {n(min_in)}: at least {n(FAIL_STEP_MIN)} tokens, "
      f"with {n(CEIL_SAMPLE - last.prompt_tokens)} left under the ceiling")
print(f"  its read_file reply: {n(last.read_content_chars)} characters of file text (truncated by the harness: "
      f"{last.read_truncated}), {n(last.obs_chars)} as the JSON reply recorded in the trace (fields and escape sequences)")
step_chars = dev.obs_chars + dev.args_chars
print(f"  tool call + reply: {n(fail_chars)} characters, the largest of any of the {len(dev)} answered calls "
      f"(next largest {n(step_chars.drop(index=last.name).max())})")
assert step_chars.max() == fail_chars

X = np.c_[np.ones(len(st)), st.chars, st.completion_tokens]
coef, *_ = np.linalg.lstsq(X, st.step.values, rcond=None)
print(f"\nprompt growth per step = {coef[0]:.1f} + {coef[1]:.3f} x (characters of tool call + tool reply) "
      f"{coef[2]:+.3f} x (output tokens of the call)")
print("  -> the tool-call text and the reply are carried into the next prompt; the rest of the output (thinking) is not")
big = st[st.chars >= 2_000].copy()
big["tokens_per_char"] = big.step / big.chars
DENSE = rec("dense", round(big.tokens_per_char.max(), 3))
med_rate = big.tokens_per_char.median()
print(f"tokens per character over the {len(big)} steps with at least 2,000 characters: median "
      f"{med_rate:.3f}, densest {DENSE:.3f}; the refused step: at least {FAIL_STEP_MIN / fail_chars:.3f}")
print(f"a full read_file (10,000 characters) at the densest rate: about {n(10_000 * DENSE)} tokens, "
      f"{10_000 * DENSE / MARGIN_SAMPLE:.1f} times the {n(MARGIN_SAMPLE)}-token room")
ZHUKOV = "136,000 and 259,000 characters"   # COPIED: Oleksii Zhukov's walkthrough, search_similar_code results
print(f"reported elsewhere (COPIED): Oleksii Zhukov finds no length cap on search_similar_code, with results of {ZHUKOV}")
print(f"  at the median rate {med_rate:.3f}, 136,000 characters is about {round(136_000 * med_rate, -3):,.0f} tokens, "
      f"more than the whole {n(CONTEXT)}-token context")
PRVSIYAN = "you requested 2048 output tokens and your prompt contains at least 30721 input tokens"   # COPIED
cap_p, in_p = map(int, re.search(r"requested (\d+) output tokens and your prompt contains at least (\d+)", PRVSIYAN).groups())
print(f"reported elsewhere (COPIED): prvsiyan's agent was refused at a {n(cap_p)} cap with a prompt of at least {n(in_p)}")
think = dev.thinking_chars.sum()
print(f"generated text: {100 * think / (think + dev.args_chars.sum()):.0f}% thinking, the rest tool-call arguments "
      f"({n(think)} vs {n(dev.args_chars.sum())} characters)")

In [7]:
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.hist(st.step, bins=np.arange(-500, 11_001, 250), color="#34495e", alpha=0.85)
top = ax.get_ylim()[1]
ax.axvline(MARGIN_SAMPLE, color="#c0392b", ls="--", lw=1.4)
ax.text(MARGIN_SAMPLE + 120, top * 0.9, f"{n(MARGIN_SAMPLE)}: room left by the sample's 16,384 cap", color="#c0392b", fontsize=9)
ROOM_8192 = CONTEXT - 8_192 - COMPACTION_TRIGGER
ax.axvline(ROOM_8192, color="#27ae60", ls="--", lw=1.4)
ax.text(ROOM_8192 - 120, top * 0.9, f"{n(ROOM_8192)}: room left by 8,192", color="#27ae60", fontsize=9, ha="right")
ax.annotate("", xy=(FAIL_STEP_MIN + 1_700, top * 0.22), xytext=(FAIL_STEP_MIN, top * 0.22),
            arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.6))
ax.plot([FAIL_STEP_MIN], [top * 0.22], marker="o", color="#c0392b")
ax.text(FAIL_STEP_MIN, top * 0.27, f"the refused step: at least {n(FAIL_STEP_MIN)}", color="#c0392b", fontsize=9)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_xlim(-500, 11_000)
ax.set_xlabel("tokens one tool round trip added to the prompt")
ax.set_ylabel("steps")
ax.set_title(f"{len(st)} measured steps against the room each cap leaves above the compaction trigger")
plt.tight_layout()
plt.show()

Most steps are small; the second bump is `read_file` (all 24 steps of 1,500 to 2,100 tokens). What a step adds is the
tool-call text and the tool's reply (0.318 tokens per character); the rest of the call's output, its thinking, adds
nothing measurable (-0.020 per output token) because it is not carried into the next prompt. The harness caps a
`read_file` at 10,000 characters of file text (the fatal reply was exactly that: 10,848 characters with its JSON fields
and escape sequences) and a command's output at 5,000 characters per stream. At the densest rate measured, 0.527 tokens
per character, one full `read_file` is about 5,270 tokens, 2.6 times the 2,048 room. A step that size only kills a task
when it starts close enough to the ceiling, which is why the 3 big steps did no harm (they started at 4,721 to 6,008
tokens) and the fatal one, starting at 12,597, did.

Two things no cap fixes. Oleksii Zhukov reports no length cap on `search_similar_code`, with results of 136,000 and
259,000 characters; at the median rate the smaller one alone is about 42,000 tokens, more than the whole context. And
prvsiyan reports a refusal even at a 2,048 cap, with a prompt of at least 30,721 tokens.

## 4. The fix: leave room for one step

A call carries at most about `COMPACTION_TRIGGER` + one step, and it must stay under `32,768 - max_output_tokens`. So
the cap has two limits: small enough to leave room for the largest step, large enough for the thinking budget plus a
tool call. The next cell checks your `MAX_OUTPUT_TOKENS` against both, prints the turn-depth view (what the prompt
would do without compaction), and replays every traced call under ten caps: which answered calls a cap would have
refused, what happens to the refused call, and how many measured outputs it would have cut.

In [8]:
# The rule: history is compacted only after a call's prompt reaches COMPACTION_TRIGGER, so a call can carry at most
# about COMPACTION_TRIGGER + one step. That has to stay under CONTEXT - max_output_tokens.
largest = LARGEST_STEP_TOKENS or int(round(fail_chars * DENSE))
longest_call_chars = int(dev.args_chars.max())
tool_call_tokens = int(round(coef[1] * longest_call_chars))


def plan(max_out, thinking=THINKING_BUDGET, step=largest):
    ceiling = CONTEXT - max_out
    room = ceiling - COMPACTION_TRIGGER
    return ceiling, room, room >= step, max_out >= thinking + tool_call_tokens


ceiling, room, fits, has_room = plan(MAX_OUTPUT_TOKENS)
lo_cap = max(int(dev.completion_tokens.max()), THINKING_BUDGET + tool_call_tokens)
hi_cap = CONTEXT - COMPACTION_TRIGGER - largest
src = ("your LARGEST_STEP_TOKENS" if LARGEST_STEP_TOKENS else
       f"the fatal step's tool call + reply, {n(fail_chars)} characters, at the densest rate {DENSE}: an estimate, not a bound")
print(f"largest step assumed: {n(largest)} tokens ({src})")
print(f"your max_output_tokens {n(MAX_OUTPUT_TOKENS)}: prompt ceiling {n(ceiling)}, room above the trigger {n(room)} "
      f"-> {'OK' if fits else 'TOO SMALL'} for a {n(largest)}-token step ({room / largest:.1f}x)")
print(f"output room for a {n(THINKING_BUDGET)}-token thinking budget plus the longest tool call "
      f"({n(longest_call_chars)} characters, about {n(tool_call_tokens)} tokens): {'OK' if has_room else 'TOO SMALL'}")
print(f"caps the rule allows with these inputs: {n(lo_cap)} to {n(hi_cap)}")
print(f"not a guarantee: a step of {n(fail_chars)} characters at 1 token per character, starting at the trigger, "
      f"would reach {n(COMPACTION_TRIGGER + fail_chars)} (ceiling at your cap: {n(ceiling)})")

first = int(per.first_prompt.median())
mean_step = float(st.step.mean())
print(f"\nturn depth: first prompt {n(first)} (median), mean step {n(mean_step)} tokens")
print(f"  without compaction, call {MAX_TURNS} would carry about {n(first + (MAX_TURNS - 1) * mean_step)} tokens; "
      f"the {n(ceiling)} ceiling would be reached near call {1 + (ceiling - first) / mean_step:.0f}")
print(f"  with compaction, the prompt stays below about {n(COMPACTION_TRIGGER + largest)} as long as each compaction "
      f"lands as low as the four seen here ({n(ev.prompt_after.min())} to {n(ev.prompt_after.max())}); observed peak "
      f"{n(dev.prompt_tokens.max())}")

fail_est = int(round(last.prompt_tokens + fail_chars * DENSE))     # the refused call at the densest measured rate
fail_stress = int(last.prompt_tokens + fail_chars)                 # ... at 1 token per character
rows = []
for cap in (1_024, 2_048, 4_096, 6_144, 8_192, 10_240, 12_288, 14_336, 16_384, 18_432):
    ceil_ = CONTEXT - cap
    over_ = dev[dev.prompt_tokens > ceil_]
    hit = set(over_.task)
    if min_in > ceil_:
        status = "refused"
        hit.add(DEAD)
    elif fail_stress <= ceil_:
        status = "fits, even at 1 token/char"
    elif fail_est <= ceil_:
        status = "fits at the densest rate"
    else:
        status = "unclear"
    rows.append({"max_output_tokens": cap, "prompt_ceiling": ceil_, "room_above_trigger": ceil_ - COMPACTION_TRIGGER,
                 "answered_calls_refused": len(over_), "the_refused_call": status,
                 "tasks_hit": len(hit), "outputs_cut": int((dev.completion_tokens > cap).sum())})
replay = pd.DataFrame(rows)
print(f"\nreplay over the {len(dev)} answered calls and the refused one (at least {n(min_in)} tokens; "
      f"about {n(fail_est)} at the densest rate; {n(fail_stress)} at 1 token per character):")
print(replay.to_string(index=False))
ok_dense = replay[(replay.tasks_hit == 0) & replay.the_refused_call.str.startswith("fits")].max_output_tokens.max()
ok_stress = replay[(replay.tasks_hit == 0) & (replay.the_refused_call == "fits, even at 1 token/char")].max_output_tokens.max()
print(f"largest cap in the replay with no refusal: {n(ok_dense)} at the densest rate, {n(ok_stress)} even at 1 token/char")
print(f"largest measured output: {n(dev.completion_tokens.max())} tokens, so any cap under it cuts at least one output")
replay.to_csv(WORK / "gemma4_cap_replay.csv", index=False)

Read the replay from the bottom. At 16,384 the sample loses 1 task; one row further (18,432) and 6 of 7 are hit. The
traces pass every cap up to 14,336 at the densest rate, and every cap up to 8,192 even at 1 token per character. Caps
under 3,088, the largest output, start cutting outputs (5 at 2,048, 17 at 1,024). The rule above, with the densest-rate
estimate of the fatal step, allows 4,663 to 12,691; 8,192 sits inside that range with room on both sides, and with the
thinking budget at 4,096 it keeps the full budget plus a tool call. It is not a guarantee: the fatal step's 10,893
characters at 1 token per character, starting right at the trigger, would reach 25,229.

## 5. The time budget

The 12-hour arithmetic is covered well already ([Abhinav Singh's calculator](https://www.kaggle.com/code/officialabhinavsingh/gemma-4-swe-task-map-12-hour-budget),
[Pathfinder](https://www.kaggle.com/code/mizeroluckygall/pathfinder-gemma-4-agent-eda-baseline)); here it is with the
measured costs. The public split has 58 tasks, so the hidden set is 116 if "evenly divided" is exact. The table gives
upper limits under an assumed overrun per task, not a recommendation: one task here ran 76.0 s past its budget, partly
in tests that the limit excludes (the split is not measured), and this evaluator's subprocess sandbox says nothing about
the scorer's Docker setup time, which the limit includes.

In [9]:
EVAL_PAGE = ("Your agent has a limit of 12 hours to submit patches for all tasks, inclusive of sandbox setup time, "
             "but excluding time for patch validation.")
DATA_PAGE = "There are about 120 tasks in the test set, evenly divided between the public and private splits."
print(f'Evaluation page (COPIED): "{EVAL_PAGE}"')
print(f'Data page (COPIED): "{DATA_PAGE}"')
HOURS = int(re.search(r"limit of (\d+) hours", EVAL_PAGE).group(1))
limit_s = HOURS * 3600
ENGINE_LOG = {   # COPIED from my run logs: [seconds since start] line; the vLLM cell runs between the two lines
    "7-task run (no adapters)": [(104.31, "Induced subgraph over 4 focal nodes: 4 nodes, 6 edges"),
                                 (454.37, "vLLM server started on http://127.0.0.1:8000/v1 (tp=4)")],
    "starter run (2 placeholder adapters)": [(98.19, "Induced subgraph over 4 focal nodes: 4 nodes, 6 edges"),
                                             (516.62, "vLLM server started on http://127.0.0.1:8000/v1 (tp=4)")],
}
ENGINE_S = {}
for k_, ((t0_, l0_), (t1_, l1_)) in ENGINE_LOG.items():
    ENGINE_S[k_] = t1_ - t0_
    print(f"{k_}: [{t0_:.2f} s] {l0_}  /  [{t1_:.2f} s] {l1_}  ->  vLLM start {ENGINE_S[k_]:.1f} s")
engine = max(ENGINE_S.values())
budget_s = 60 * MAX_TIME_MINUTES
tb = T[T.outcome == "time budget (4 min)"]
beyond = (tb.wall_s - budget_s).sort_values()
print(f"\n{HOURS} h = {limit_s:,} s; per task over {N_HIDDEN_TASKS} tasks: {limit_s / N_HIDDEN_TASKS:.1f} s")
print(f"tasks that ran out the {MAX_TIME_MINUTES}-minute budget ended {', '.join(f'{v:.1f}' for v in beyond)} s after it "
      f"(setup, the call in flight, patch extraction and, for patched tasks, the tests, which the limit excludes; "
      f"the split is not measured)")
rows = []
for n_tasks in (N_HIDDEN_TASKS, 120):
    for allow in (10, 60):
        fit = (limit_s - engine) / n_tasks - allow
        rows.append({"tasks": n_tasks, "assumed_overrun_s_per_task": allow,
                     "max_time_minutes_at_most": round(fit / 60, 2)})
print(pd.DataFrame(rows).to_string(index=False))
used = N_HIDDEN_TASKS * (budget_s + 10) + engine
print(f"with max_time_minutes = {MAX_TIME_MINUTES}: about {used / 3600:.1f} h of the {HOURS} h, one task at a time")

span = dev.groupby("task").tool_done_s.max()
model_share = dev.groupby("task").model_s.sum() / span
tool_total = dev.groupby("task").tool_s.sum()
print(f"\nwhere the agent time went: model calls {100 * model_share.min():.1f}% to {100 * model_share.max():.1f}% of it; "
      f"running tools {tool_total.min():.1f} to {tool_total.max():.1f} s per task")
print(f"compaction: {len(ev)} drops, {ev.seconds_beyond_output.sum():.0f} s in total "
      f"({ev.seconds_beyond_output.min():.0f} to {ev.seconds_beyond_output.max():.0f} s each)")

## 6. What one task costs on 4 x L4

One task at a time, from the run log, the traces and `nvidia-smi`. The memory figure is what vLLM reserves at start
(`gpu_memory_utilization` 0.90 here, 0.80 on the scorer), not what a task uses. Output speed counts from the previous
tool result to the model's answer, so it includes prefill; the headline rate leaves out the 2 calls that also paid for
a compaction.

In [10]:
LOG = {   # COPIED from my run log (vLLM 0.19.1, tensor parallel 4, gpu_memory_utilization 0.90)
    "load": "Model loading took 5.96 GiB memory and 70.199928 seconds",
    "kv": "GPU KV cache size: 46,352 tokens",
    "conc": "Maximum concurrency for 32,768 tokens per request: 8.74x",
}
for v_ in LOG.values():
    print("log:", v_)
g = gpu.groupby("gpu").agg(peak=("mem_used_mib", "max"), total=("mem_total_mib", "max"), util=("util_pct", "mean"))
base_long = base[base.completion_tokens >= 300]
cost = pd.DataFrame([
    ("vLLM start", f"{ENGINE_S['7-task run (no adapters)']:.1f} s (no adapters); "
                   f"{ENGINE_S['starter run (2 placeholder adapters)']:.1f} s (starter, 2 placeholder adapters)"),
    ("weights per GPU", re.search(r"took ([\d.]+ GiB)", LOG["load"]).group(1)),
    ("KV cache", re.search(r"size: ([\d,]+ tokens)", LOG["kv"]).group(1) + " at gpu_memory_utilization 0.90"),
    ("GPU memory in use", f"{int(g.peak.max()):,} of {int(g.total.max()):,} MiB per L4 (reserved at start)"),
    ("GPU utilization during tasks", f"{g.util.min():.1f} to {g.util.max():.1f}% mean per GPU ({len(gpu):,} samples, 5 s apart)"),
    ("output speed (prefill included)", f"{RATE} tokens/s over {len(clean)} calls with >= 300 output tokens; "
                                        f"{RATE_ALL} with the {len(long_calls) - len(clean)} that also paid for a compaction"),
    ("output speed, starter run", f"{base_long.completion_tokens.sum() / base_long.model_s.sum():.1f} tokens/s over "
                                  f"{len(base_long)} calls (placeholder adapters loaded)"),
    ("model calls per task", f"{int(T.model_calls.min())} to {int(T.model_calls.max())} (median {int(T.model_calls.median())})"),
    ("prompt per call", f"median {n(dev.prompt_tokens.median())}, max {n(dev.prompt_tokens.max())} tokens"),
    ("output per call", f"median {n(dev.completion_tokens.median())}, max {n(dev.completion_tokens.max())} tokens"),
    ("prompt tokens per task", f"{n(T.prompt_tokens_total.min())} to {n(T.prompt_tokens_total.max())}"),
    ("output tokens per task", f"{n(T.output_tokens.min())} to {n(T.output_tokens.max())} (compaction summaries not counted)"),
    ("wall time per task", f"{T.wall_s.min()} to {T.wall_s.max()} s at a {MAX_TIME_MINUTES}-minute budget"),
    ("compaction", f"{len(ev)} in {int((T.compactions > 0).sum())} of {len(T)} tasks, "
                   f"{ev.seconds_beyond_output.min():.0f} to {ev.seconds_beyond_output.max():.0f} s each"),
], columns=["quantity", "measured"])
pd.set_option("display.max_colwidth", 130)
print(cost.to_string(index=False))
cost.to_csv(WORK / "gemma4_cost_table.csv", index=False)

## 7. Limits

* **Seven tasks.** 1 of 7 has an exact 95% interval of 0.4% to 57.9%; do not read it as the scorer's rate. The step
  sizes (97 steps) carry the mechanism, and the rule itself is vLLM's arithmetic, not a sample.
* **This evaluator is not the scorer.** It is the official starter's evaluator, which differs from the scorer's
  documented settings in four places (table below). The trigger (14,336) and the context (32,768) are the same. The
  vLLM version (0.19.1 here) and the scorer's handling of a refusal are not documented: the task ended there in this
  evaluator. This run's 32-call task never compacted on its interval of 15, so the interval does not count model calls;
  whether the scorer's interval of 5 fires is not measured.
* **One task at a time.** Speed and time per task at higher concurrency are not measured.
* **The refused prompt's size is a lower bound.** vLLM reports "at least 16385"; every statement about that call is a
  bound or an estimate at a stated rate.
* **What the harness keeps when a task dies after an edit** is not visible here (the dead task had not edited).
  Oleksii Zhukov reads the harness code as discarding the patch.
* **The fix is replayed, not rerun.** For every answered call the replay is arithmetic (its prompt against the
  ceiling, its output against the cap); how the refused task would have continued is unknown.

In [11]:
diff = pd.DataFrame([
    ("gpu_memory_utilization", "0.90", "0.80"),
    ("enable_thinking chat-template default", "not set", "True"),
    ("compaction_interval", "15", "5"),
    ("sandbox", "subprocess", "Docker (README section 4.1)"),
    ("token_threshold (compaction trigger)", "14,336", "14,336"),
    ("max_model_len", "32,768", "32,768"),
    ("vLLM version", "0.19.1", "not documented"),
    ("handling of the refusal", "task ends, no patch", "not documented"),
], columns=["setting", "this run (official starter's evaluator)", "scorer (HARNESS_README)"])
print(diff.to_string(index=False))
r4077 = dev[dev.task == T.model_calls.idxmax()]
first_drop = int(r4077[r4077.compacted_next].call.min()) + 1
print(f"\n{T.model_calls.idxmax()}: {int(T.model_calls.max())} model calls, first drop at call {first_drop}, after a prompt of "
      f"{n(r4077[r4077.call == first_drop - 1].prompt_tokens.iloc[0])}; no drop in its first {first_drop - 1} calls, "
      f"so an interval of 15 {'did' if first_drop - 1 <= 15 else 'did not'} fire by model-call count")
print(f"1 of 7 has an exact 95% interval of {HL['ci_lo']}% to {HL['ci_hi']}%; the step sizes come from {len(st)} steps")

## Credits and links

* [Getting Started - Gemma 4 Developer Agent](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent)
  by **Ryan Holbrook**: the evaluator, the vLLM setup and the base config that ran here.
* [Gemma 4 | Walkthrough & First Submission](https://www.kaggle.com/code/zhukovoleksiy/gemma-4-walkthrough-first-submission)
  by **Oleksii Zhukov**: the prompt limit rule (32,768 minus `max_output_tokens`), the uncapped `search_similar_code`
  results, and the forum finding that thoughts are not passed between tool calls, which the step sizes here agree with.
* [GEMMA: EDA, Baseline for a start](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1)
  by **Roman Rozen**: the context-budget chart and the 8,192 cap in the top public bundle.
* [Gemma and the Shape of Doubt](https://www.kaggle.com/code/prvsiyan/gemma-and-the-shape-of-doubt) by **prvsiyan**:
  the 24,576-token arithmetic for an 8,192 reserve and the refusal at a 2,048 cap.
* [Black Cat SWE Agent | Pack Instinct](https://www.kaggle.com/code/lucifer19/black-cat-swe-agent-pack-instinct) by
  **Agent Trainer** (lucifer19): the file-by-file comparison that documents the top bundle's settings.
* [Gemma 4 SWE: Task Map & 12-Hour Budget](https://www.kaggle.com/code/officialabhinavsingh/gemma-4-swe-task-map-12-hour-budget)
  by **Abhinav Singh** and [Pathfinder](https://www.kaggle.com/code/mizeroluckygall/pathfinder-gemma-4-agent-eda-baseline)
  by **Mizero Lucky Gall**: the 12-hour arithmetic.
* On reading the leaderboard in tasks rather than decimals: [the 58-task leaderboard notebook](https://www.kaggle.com/code/dariushafshar/gemma-4-lb-58-tasks-bronze-is-a-124-team-tie).

The last cell writes the per-call table, the per-task table, the cap replay and the cost table as CSV files.

In [12]:
out = dev[["task", "call", "t_s", "tool_done_s", "prompt_tokens", "completion_tokens", "tool", "args_chars", "obs_chars",
           "thinking_chars", "read_content_chars"]]
out.to_csv(WORK / "gemma4_calls_per_task.csv", index=False)
T[cols + ["first_prompt", "prompt_tokens_total"]].to_csv(WORK / "gemma4_tasks.csv")
HL.update({"steps": len(st), "steps_over_margin": len(over), "largest_step": int(st.step.max()), "fail_est": fail_est,
           "fail_stress": fail_stress, "hi_cap": hi_cap, "lo_cap": lo_cap, "rate_all": RATE_ALL})
(WORK / "headline.json").write_text(json.dumps(HL, indent=1, default=str))
for p in sorted(WORK.glob("gemma4_*.csv")):
    print(f"wrote {p.name} ({p.stat().st_size:,} bytes)")
print(f"notebook ran in {time.time() - T0:.1f} s")